In [13]:
from fuzzy_evaluation import getClass, getF1_threshold
import os
import pandas as pd
import glob
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from IPython.display import display
import numpy as np

def createConfusionMatrix_threshold(predictions, class_list, class_list_single):
    if 'type' in predictions.columns:
        targets = predictions.loc[predictions["source"] == "target", ].reset_index(drop=True)
        predictions = predictions.loc[predictions["type"] == "prediction", ].reset_index(drop=True)
    else:
        targets = predictions.loc[predictions["source"] == "target", ].reset_index(drop=True)
        predictions = predictions.loc[predictions["source"] == "prediction", ].reset_index(drop=True)

    test = predictions.merge(targets, on="sample_id", suffixes=('_pred', '_true'))

    # Create an array to store thresholds for each crop
    thresholds = np.full(len(class_list_single), 0.5)

    def update_plot(**kwargs):
        # Update thresholds based on slider values
        for i, crop in enumerate(class_list_single):
            thresholds[i] = kwargs[crop]

        # Clear the current plot
        plt.clf()

        # Update target and predicted classes based on the thresholds
        test["target_class"] = getClass(test, class_list, source="target", threshold=0)
        test["predicted_class"] = getClass(test, class_list, source="prediction",threshold=thresholds)

        # Calculate F1 score
        F1 = getF1_threshold(predictions, class_list, threshold=dict(zip(class_list_single, thresholds)))

        # Get all unique classes from both target and predicted columns
        all_classes = sorted(set(test["target_class"].unique()).union(test["predicted_class"].unique()))

        # Create confusion matrix
        cm = confusion_matrix(test["target_class"], test["predicted_class"], labels=all_classes)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=all_classes)
        disp.plot(xticks_rotation='vertical')

        # Add title with threshold values
        plt.title(f'Confusion Matrix: F1: {F1:.2f}')
        plt.show()

    # Create sliders for each crop type
    sliders = {
        crop: widgets.FloatSlider(
            value=0.25,
            min=0.0,
            max=1.0,
            step=0.01,
            description=f'{crop}:',
            continuous_update=False
        )
        for crop in class_list_single
    }

    # Use `interact` to link sliders to the update_plot function
    interact(update_plot, **sliders)


main_folder = "/vitodata/worldcereal/data/COP4GEOGLAM/mozambique/"

nc_folder = os.path.join(main_folder,"production","v3_landcover","raw")
folder = os.path.join(main_folder,"fuzzy_test")
os.makedirs(folder, exist_ok=True)

pred_file = glob.glob(os.path.join(folder, 'predictions_presto_run=202510151422.parquet'))[0]

predictions = pd.read_parquet(pred_file)

class_list = [
            "maize",
            "rice",
            "soybean",
            "sesame",
            "cassava",
            "cowpea",
            "sweet_potato",
            "pigeon_pea",
            "sugarcane",
            "other",
            "maizexcassava",
            "cassavaxpigeon_pea",
            "maizexcassavaxpigeon_pea",
    ]

class_list_single = [
            "maize",
            "rice",
            "soybean",
            "sesame",
            "cassava",
            "cowpea",
            "sweet_potato",
            "pigeon_pea",
            "sugarcane",
            "other",
    ]

#create a slider to determine the threshold



createConfusionMatrix_threshold(predictions, class_list,class_list_single)

interactive(children=(FloatSlider(value=0.25, continuous_update=False, description='maize:', max=1.0, step=0.0…